# SPAR Report — Three-state latent indicators (GWT)

**Iteration:** joint multi-system baseline + three-state leaf model.  
**Scientific framing:** this branch is a **candidate library member** alongside the validated binary baseline, not a replacement. The three-state leaf is a low-rank approximation to graded indicator strength; we do not claim indicators are literally three-state. 

## Iteration ledger (week-6 baseline + branches)

| Branch | Status | What it showed |
|---|---|---|
| Joint shared-κ baseline | Validated reporting baseline | Correct ordering Human ≫ Chicken > LLM ≫ ELIZA, but stratified PPC reveals wrong-tail misfit at Derek×{Human,ELIZA} and middle-over-spread at Rachael×Chicken / Luhan×LLMs. |
| Tight expert shifts (σ=0.3) | Rejected | Did not substantively fix the stratified PPC failure. Non-overlapping expert pools absorb system-level signal when shifts are free. |
| Wide-prior control | Null | Ordering preserved, shared $a$ essentially unchanged, stratified PPC essentially unchanged. The residual misfit is **not** baseline prior tightness. |
| Hierarchical expert cutpoints | Mixed diagnostic | Fixes Rachael / Luhan middle-heavy cells, partially improves Derek×Human, but introduces wrong-tail failures at Derek×ELIZA and a cat-7 spike across LLMs. Rating-scale flexibility matters, but expert calibration alone is not the clean solution. |
| Reference-system recovery (Stage 1 post-hoc) | Substantive finding | Under Beta(1,5), Human/ELIZA do not self-recover near-extreme; anchors are doing work the current single-expert reference data cannot corroborate. |
| **Three-state leaf (this notebook)** | Under evaluation | Minimum structural change targeting the binary-$z$ bottleneck identified in week 6. |

## What this notebook does

1. **Stage 1** — NumPy-only predictive-shape sweep at plausible $(a, \kappa)$. No MCMC.
2. **Stage 2** — Response-style synthetic gate. If the three-state model silently "fixes" response-style heterogeneity, it is absorbing misspecification we did not intend.
3. **Stage 0 / 1b / 5a** — Real-data diagnostics. Requires a joint GWT baseline fit in the same session (set `EXECUTE_FITS = True`).
4. **Stage 5b** — Beta-latent diagnostic + PSIS-LOO comparison. Non-blocking follow-up; stub only.
5. **Stage 6** — Interpretation template drafted before fits; numbers filled in afterwards.

In [ ]:
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name == "notebooks":
    PROJECT_ROOT = NB_DIR.parent
else:
    PROJECT_ROOT = NB_DIR
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "figure.constrained_layout.use": True})

from gwt_three_state_indicator_analysis import (
    FOCUS_KEYS,
    SYSTEM_CONFIGS_VALIDATED,
    SYSTEM_DISPLAY,
    abstract_shape_sweep,
    compare_c_summaries,
    emission_separation_l1,
    emission_separation_l1_from_posterior,
    fit_gwt,
    format_c_comparison_table,
    format_regime_table,
    frozen_theta_three_state_ppc,
    generate_response_style_synthetic_tree,
    middle_state_utilisation_per_cell,
    plot_shape_sweep,
    posterior_mean_a_kappa,
    regime_rows_for_focus_cells,
)
from dcm_model import (
    BayesianModelBuilder,
    EvidenceProcessor,
    ModelConfig,
    MultiSystemDataProcessor,
    MultiSystemModelBuilder,
    OrdinalDataProcessor,
)
from dcm_ppc import (
    per_expert_system_ppc_multisystem,
    per_expert_system_tree_implied_ppc_multisystem,
    plot_per_expert_system_histograms,
)

# Flip to True to kick off the full joint-GWT fits (~75 minutes each on CPU).
EXECUTE_FITS = False

print(f"project root: {PROJECT_ROOT}")
print(f"EXECUTE_FITS = {EXECUTE_FITS} (Stage 0 / 1b / 5a require True)")

## Stage 1 — NumPy-only predictive-shape check

Does the three-state leaf admit:
- strong $P(r{=}1)$ at small $q$?
- strong $P(r{=}7)$ at large $q$?
- middle-heavy predictions at moderate $q$, **without** degrading extreme-category mass?

We evaluate both leaf models at plausible $(a, \kappa)$ drawn from the independent-LLM summary ($a \approx 1.45$ for GWT under the ordinal fit). Under joint fits, $a$ was reported at ${\approx}1.9$ in week 6, so we also sweep that value.

**Acceptance signal.** The three-state signed-tail curves should track binary at extremes and add middle-mass only at moderate $q$. Wrong-tail degradation (3-state $P(r{=}7)$ drops below binary at $q{\approx}1$) would be a red flag before we fit anything.

In [ ]:
# Plausible baseline cutpoints and two candidate discrimination values.
kappa_baseline = np.array([-1.5, -0.8, -0.1, 0.4, 1.0, 1.6])

for a_val in (1.45, 1.91):
    sweep = abstract_shape_sweep(a_val, kappa_baseline)
    tails = sweep.tail_curves()
    print(f"\na = {a_val}")
    print(f"{'q':>6s} {'bin P(r=1)':>12s} {'3st P(r=1)':>12s} "
          f"{'bin P(r=7)':>12s} {'3st P(r=7)':>12s} "
          f"{'bin P(mid)':>12s} {'3st P(mid)':>12s}")
    for i, q in enumerate(sweep.q_grid):
        print(
            f"{q:>6.2f} {tails['binary_left'][i]:>12.3f} {tails['three_state_left'][i]:>12.3f} "
            f"{tails['binary_right'][i]:>12.3f} {tails['three_state_right'][i]:>12.3f} "
            f"{tails['binary_mid'][i]:>12.3f} {tails['three_state_mid'][i]:>12.3f}"
        )
    fig, _ = plot_shape_sweep(sweep, title=f"Leaf-model shape sweep (a = {a_val})")
    plt.show()

## Stage 2 — Response-style synthetic gate (required)

Generate synthetic ratings with **binary underlying presence** but **expert-specific cutpoint jitter and scale** (classical response-style heterogeneity). Fit both binary and three-state. The three-state should NOT appear to substantively "fix" this misspecified process — if it does, any fit improvement on real GWT data might reflect the same kind of absorption, not genuine graded presence.

The test uses short fits (NUTS-fast settings) to keep this gate cheap.

In [ ]:
tree, _, sys_name, meta = generate_response_style_synthetic_tree(
    n_features=3, indicators_per_feature=4, n_experts=4,
    true_a=1.8, cutpoint_loc_sigma=0.35, cutpoint_scale_sigma=0.2,
    generator_model="binary", seed=42,
)
print(f"synthetic system: {sys_name}")
print(f"expert locs: {meta['expert_locs']}")
print(f"expert scales: {meta['expert_scales']}")

cfg_base = dict(
    NUM_SAMPLES=400, NUM_TUNE=500, NUM_CHAINS=2,
    TARGET_ACCEPT=0.9, USE_EXPERT_SHIFTS=False,
)

per_cell_results = {}
for state_model in ("binary", "three_state"):
    cfg = ModelConfig(INDICATOR_STATE_MODEL=state_model, **cfg_base)
    processor = OrdinalDataProcessor(cfg)
    processor.process(tree, sys_name)
    builder = BayesianModelBuilder(cfg, EvidenceProcessor(cfg), processor)
    model = builder.build_model(tree)
    idata = builder.sample(model)
    per_cell_results[state_model] = {
        "idata": idata,
        "builder": builder,
        "processor": processor,
        "a_mean": float(idata.posterior["a"].values.mean()),
    }
    print(f"  {state_model}: a posterior mean = {per_cell_results[state_model]['a_mean']:.3f}")

In [ ]:
# Quick qualitative check: posterior-mean a under response-style-heterogeneous binary-generated
# data. If the three-state branch drives a materially higher than the binary branch here,
# that is evidence it is "rescuing" itself from the heterogeneity rather than reflecting a
# genuinely three-state process.
a_delta = per_cell_results["three_state"]["a_mean"] - per_cell_results["binary"]["a_mean"]
print(f"Stage 2 response-style gate -- true a = 1.80")
print(f"  binary posterior-mean a      = {per_cell_results['binary']['a_mean']:.3f}")
print(f"  three-state posterior-mean a = {per_cell_results['three_state']['a_mean']:.3f}")
print(f"  delta (three_state - binary) = {a_delta:+.3f}")
print()
if abs(a_delta) < 0.25:
    print("OK: three-state did not substantively shift discrimination relative to binary")
    print("    on binary-with-response-style-heterogeneity data. Proceed to Stage 5a.")
else:
    print("FLAG: three-state shifts a materially on response-style-only synthetic data.")
    print("      Investigate before promoting as a library candidate.")

## Stage 0 / 1b / 5a — Real-data fits

These stages require the validated joint-GWT baseline to be fit (binary state model). Each full fit takes ~75 minutes on CPU at default sampling settings.

Flip `EXECUTE_FITS = True` at the top of the notebook to proceed. A truncated-sample mode (`FAST_FIT = True` below) runs a short pilot fit for code validation; it is not a substitute for the full fit when reporting results.

In [ ]:
FAST_FIT = True  # toggles short pilot sampling for code validation
fit_overrides_fast = dict(NUM_SAMPLES=300, NUM_TUNE=400, NUM_CHAINS=2, TARGET_ACCEPT=0.9)

binary_result = None
three_state_result = None

if EXECUTE_FITS:
    overrides = fit_overrides_fast if FAST_FIT else None
    binary_result = fit_gwt(state_model="binary", fit_overrides=overrides)
    three_state_result = fit_gwt(state_model="three_state", fit_overrides=overrides)
    print("\nBinary vs three-state C posteriors:")
    cmp = compare_c_summaries(binary_result, three_state_result)
    print(format_c_comparison_table(cmp, guardrail=0.05))
else:
    print("Stage 0 / 1b / 5a skipped -- set EXECUTE_FITS = True to run.")
    print("Expected cost: ~75 minutes per fit at default settings (two fits needed).")

In [ ]:
# --- Stage 0: q_j regime table for focus cells under the binary baseline ---
if binary_result is not None:
    rows = regime_rows_for_focus_cells(
        binary_result["idata"], binary_result["builder"], binary_result["processor"]
    )
    print("STAGE 0 -- focus-cell q_j regimes (binary baseline):\n")
    print(format_regime_table(rows))

In [ ]:
# --- Stage 1b: frozen-theta counterfactual PPC ---
if binary_result is not None:
    frozen_strat = frozen_theta_three_state_ppc(
        binary_result["idata"], binary_result["builder"], binary_result["processor"],
        n_draws=300, seed=0,
    )
    print("STAGE 1b -- frozen-theta three-state PPC at focus cells:")
    for key in FOCUS_KEYS:
        r = frozen_strat.get(key)
        if r is None:
            continue
        print(
            f"  {key[0]} x {SYSTEM_DISPLAY.get(key[1], key[1]):<12s} "
            f"obs_mean={r['obs_mean']:.2f} pred={r['pred_mean_mean']:.2f}  "
            f"\u0394left={r['delta_left_mean']:+.2f} \u0394right={r['delta_right_mean']:+.2f} "
            f"\u0394mid={r['delta_mid_mean']:+.2f}"
        )

In [ ]:
# --- Stage 5a: real-data three-state PPC + emission-separation L1 ---
if three_state_result is not None:
    ppc_strat = per_expert_system_ppc_multisystem(
        three_state_result["idata"],
        three_state_result["builder"],
        three_state_result["processor"],
        n_draws=500,
        seed=0,
    )
    print("STAGE 5a -- three-state PPC at focus cells:")
    for key in FOCUS_KEYS:
        r = ppc_strat.get(key)
        if r is None:
            continue
        print(
            f"  {key[0]} x {SYSTEM_DISPLAY.get(key[1], key[1]):<12s} "
            f"obs_mean={r['obs_mean']:.2f} pred={r['pred_mean_mean']:.2f}  "
            f"\u0394left={r['delta_left_mean']:+.2f} \u0394right={r['delta_right_mean']:+.2f} "
            f"\u0394mid={r['delta_mid_mean']:+.2f}"
        )

    l1 = emission_separation_l1_from_posterior(
        three_state_result["idata"], threshold=0.1,
    )
    print("\nEmission-separation L1 (posterior means):")
    for k, v in l1.items():
        print(f"  {k}: {v}")

    util = middle_state_utilisation_per_cell(
        three_state_result["idata"],
        three_state_result["builder"],
        three_state_result["processor"],
    )
    p_m1_vals = np.array([r["p_m1_mean"] for r in util])
    print(f"\nMiddle-state utilisation E[m=1|ratings] across {len(util)} indicators:")
    print(
        f"  mean = {p_m1_vals.mean():.3f}  median = {np.median(p_m1_vals):.3f}  "
        f"3% = {np.percentile(p_m1_vals, 3):.3f}  97% = {np.percentile(p_m1_vals, 97):.3f}"
    )

In [ ]:
# --- Stratified PPC grid, three-state vs binary, for visual comparison ---
if binary_result is not None and three_state_result is not None:
    binary_strat = per_expert_system_ppc_multisystem(
        binary_result["idata"],
        binary_result["builder"],
        binary_result["processor"],
        n_draws=500, seed=0,
    )
    fig_b, _ = plot_per_expert_system_histograms(
        binary_strat, K=7, system_display=SYSTEM_DISPLAY,
        system_order=[s for s, _ in SYSTEM_CONFIGS_VALIDATED],
        expert_order=binary_result["processor"].expert_names,
        title="Stratified PPC -- binary baseline",
    )
    plt.show()
    fig_t, _ = plot_per_expert_system_histograms(
        ppc_strat, K=7, system_display=SYSTEM_DISPLAY,
        system_order=[s for s, _ in SYSTEM_CONFIGS_VALIDATED],
        expert_order=three_state_result["processor"].expert_names,
        title="Stratified PPC -- three-state leaf",
    )
    plt.show()

## Stage 5b — Beta-latent diagnostic + PSIS-LOO (TODO)

Non-blocking follow-up. Fits a Beta-latent leaf $z \sim \mathrm{Beta}(\lambda q, \lambda (1-q))$ at fixed $\lambda \in \{1, 4, 8\}$ via Gauss-Legendre quadrature over $z$, and runs PSIS-LOO across {binary, three-state, Beta-latent($\lambda$)}. 

If the three-state gain is reproduced by the Beta-latent at moderate-to-high $\lambda$, the mechanism is generic within-cell overdispersion rather than three discrete emission centres specifically — report plainly and soften the Stage 5a interpretation.

**Implementation note.** The Beta-latent fit requires a new PyTensor-graph likelihood that marginalises over $z$ with fixed Gauss-Legendre nodes. This is scoped as a follow-up commit after Stage 5a lands.

## Stage 6 — Interpretation template (draft before viewing fit results)

> We fit a three-state latent indicator leaf model ($m_j \sim \mathrm{Binomial}(2, q_j)$, emission centres at $\eta \in \{0, a/2, a\}$) on the validated joint GWT baseline and compared it to the current binary leaf model. **System ordering** remained Human $\gg$ Chicken > LLM $\gg$ ELIZA: [FILL]. **Chicken and LLM posterior medians** moved by $|\Delta| = $ [FILL], [FILL] relative to binary, within / outside the pre-registered $\pm 0.05$ guardrail. **Stratified PPCs** showed [improved / unchanged / worse] shape-correct behaviour at the focus cells, with the signed-tail diagnostic [improving / showing a wrong-tail spike] at Derek$\times$ELIZA and Derek$\times$Human. The **emission-separation L1 diagnostic** reported $\|g(0) - g(a/2)\|_1 = $ [FILL] and $\|g(a/2) - g(a)\|_1 = $ [FILL]; the middle emission [was / was not] empirically distinct from either endpoint. **Middle-state utilisation** $\mathbb{E}[m{=}1 \mid \text{ratings}]$ was [localised to middle-heavy cells / distributed uniformly across cells], consistent with [genuine graded presence / generic overdispersion absorption]. The **Stage 2 response-style gate** [passed / flagged] — on binary-generated data with expert-specific cutpoint heterogeneity, the three-state model [did not / did] silently shift discrimination. Taken together, we [append the three-state model as a library alternative / do not recommend it at this stage]; the Stage 5b Beta-latent diagnostic is sequenced next to disambiguate three discrete grades from generic within-cell overdispersion.

Do not rewrite the structure of this paragraph after seeing numbers; just fill in the [FILL] slots and remove the branches that do not apply.